In [ ]:
import arcpy
from arcpy.sa import ExtractByMask

# check out Spatial Analyst license
arcpy.CheckOutExtension("Spatial")

In [ ]:
# ── PATHS ────────────────────────────────────────────────────────────────────

# Input bathy rasters (USC and GBC rasters cropped to thesis boundary)
gbc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb\GBC15a_bathy_thesis"
usc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb\USC10m_bathy_thesis"

# Input slope rasters (USC and GBC rasters cropped to thesis boundary)
gbc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb\GBC15a_slope_thesis"
usc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb\USC10m_slope_thesis"

# Output geodatabase (same .gdb as input rasters)
out_gdb = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb"

# Output polygon feature classes (boundaries built from bathy rasters and reused for slope since identical)
usc_boundary = out_gdb + r"\boundary_usc_thesis"
gbc_boundary = out_gdb + r"\boundary_gbc_thesis"
nooverlap = out_gdb + r"\boundary_gbc_thesis_NOOVERLAP"

# Output GBC rasters ** with USC overlap area removed **
bathy_nooverlap_raster = out_gdb + r"\GBC15a_bathy_thesis_NOOVERLAP"
slope_nooverlap_raster = out_gdb + r"\GBC15a_slope_thesis_NOOVERLAP"

In [ ]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [ ]:
# ── 1. CREATE BOUNDARY POLYGON FOR EACH RASTER ───────────────────────────────
#⊹ (boundary polygons built from bathy rasters and used for slope since raster "footprints" (boundaries) are identical)
#ㅱ NOT IDENTICAL - NEED TO FIX!!!!
arcpy.ddd.RasterDomain(usc_bathy, usc_boundary, "POLYGON")
arcpy.ddd.RasterDomain(gbc_bathy, gbc_boundary, "POLYGON")

In [ ]:
# ── 2. CREATE * NO OVERLAP * GBC BOUNDARY POLYGON (ERASE USC OVERLAP) ────────

#❥ run 'Erase' tool (removes boundary_usc_thesis polygon area from boundary_gbc_thesis polygon)
arcpy.analysis.Erase(gbc_boundary, usc_boundary, nooverlap)

In [ ]:
# ── 3. CREATE * NO OVERLAP * GBC BATHYMETRY RASTER (ERASE USC OVERLAP) ──────

#❥ run 'Extract by Mask' tool (crops GBC15a_bathy_thesis raster to boundary_gbc_thesis_NOOVERLAP polygon)
out_bathy = ExtractByMask(gbc_bathy, nooverlap)

# output raster boundary_gbc_thesis_NOOVERLAP to ~\Bathymetry_geodatabases\thesis_custom_rasters.gdb
out_bathy.save(bathy_nooverlap_raster)

In [ ]:
# ── 4. CREATE * NO OVERLAP * GBC SLOPE RASTER (ERASE USC OVERLAP) ──────────

#❥ run 'Extract by Mask' tool (crops GBC15a_slope_thesis raster to boundary_gbc_thesis_NOOVERLAP polygon)
#⊹ (reuse same nooverlap polygon since slope and bathy rasters have identical footprint)
out_slope = ExtractByMask(gbc_slope, nooverlap)

# export new no overlap GBC slope raster
out_slope.save(slope_nooverlap_raster)

In [ ]:
# ── PRINT CONFIRMATION ───────────────────────────────────────────────────────
print("Done")
print("USC boundary:", usc_boundary)
print("GBC boundary:", gbc_boundary)
print("GBC with USC removed:", nooverlap)
print("GBC bathy raster with USC area removed:", bathy_nooverlap_raster)
print("GBC slope raster with USC area removed:", slope_nooverlap_raster)

# release Spatial Analyst license
arcpy.CheckInExtension("Spatial")